# 04 — Inference & Submission

Dijalankan SEKALI di Kaggle GPU oleh Lead, setelah semua tahap sebelumnya dibekukan (frozen).

**Ini adalah SATU-SATUNYA notebook di seluruh pipeline yang diizinkan membuka file dari folder
`test/`.** Tidak ada logika training di sini. `TEST_ROOT` di bawah adalah satu-satunya definisi
path data test di seluruh codebase (PRD §2 rule 3, §11).


In [ ]:
# torch/torchvision/pandas/numpy/scipy/pillow/tqdm sudah terpasang di image GPU Kaggle (dipakai
# puluhan paket bawaan lain, dan torch/torchvision di sana sudah cocok dengan driver CUDA-nya) —
# tidak di-pin exact di sini supaya tidak bentrok/merusak link CUDA (deviasi dari PRD §12, sama
# seperti di NB01/02x). Hanya `timm` dan `imagehash` yang tidak bawaan Kaggle yang di-pin persis.
!pip install -q timm==1.0.11 imagehash==4.3.1


**PENTING sebelum lanjut**: instalasi `imagehash` kadang membuat numpy/pandas yang sudah ter-load
di proses kernel jadi tidak konsisten secara biner (error umum: `numpy.dtype size changed, may
indicate binary incompatibility`). Sel di bawah ini SENGAJA mem-restart proses kernel secara paksa
supaya numpy/pandas ter-load ulang bersih dari disk — ini normal, bukan crash. Setelah reconnect,
lanjutkan dari sel import di bawah — TIDAK perlu menjalankan ulang sel pip install di atas.


In [ ]:
import os
os.kill(os.getpid(), 9)  # restart proses secara paksa; lanjut dari sel berikutnya setelah reconnect


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import timm.data
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 3
N_FOLDS = 5
print("device:", DEVICE, "| torch:", torch.__version__, "| timm:", timm.__version__,
      "| numpy:", np.__version__, "| pandas:", pd.__version__)


## Config

`TEST_ROOT` adalah satu-satunya tempat path `test/` pernah dibentuk di seluruh repo ini.
`ARCH_REGISTRY` mencerminkan keempat sel `CONFIG` di 02x (tag + drop_path_rate) supaya arsitektur
yang persis sama bisa dibangun ulang di sini untuk menerima bobot hasil fine-tune.

Set `RUNTIME_ENV` sesuai tempat notebook ini dijalankan. Struktur data mentah asli dari panitia:
`BDC2026/test/{id}.jpg` (id 1-1458 langsung, tanpa leading zero) dan `BDC2026/submission.csv`
(template resmi) — cocok di kedua environment, tinggal beda root path-nya saja.


In [ ]:
def resolve_kaggle_input_dir(slug: str) -> Path:
    # Beberapa environment Kaggle memasang dataset langsung di /kaggle/input/{slug}; environment
    # lain (mount layout baru) menaruhnya lebih dalam (mis. /kaggle/input/datasets/{slug} atau
    # .../datasets/{owner}/{slug}). Cari otomatis daripada menebak satu pola path yang kaku.
    direct = Path(f"/kaggle/input/{slug}")
    if direct.exists():
        return direct
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for dirpath, dirnames, _ in os.walk(input_root):
            if Path(dirpath).name == slug:
                return Path(dirpath)
            if len(Path(dirpath).relative_to(input_root).parts) >= 3:
                dirnames[:] = []
    raise FileNotFoundError(
        f"Dataset '{slug}' tidak ketemu di bawah /kaggle/input/ (dicoba path langsung maupun "
        f"pencarian rekursif sampai 3 level). Isi /kaggle/input/: "
        f"{sorted(p.name for p in input_root.iterdir()) if input_root.exists() else []}\n"
        f"Cek panel 'Input' di kanan notebook -- kalau '{slug}' hilang, klik 'Add Input' untuk "
        f"attach ulang, lalu 'Save Version' lagi."
    )


RUNTIME_ENV = "colab"  # "colab" (mount Drive, sama seperti NB01) atau "kaggle" (dataset di-attach)

if RUNTIME_ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path("/content/drive/MyDrive/Big Data Challenge - Satria Data 2026")
    TEST_ROOT = DRIVE_ROOT / "BDC2026" / "test"
    TEMPLATE_PATH = DRIVE_ROOT / "BDC2026" / "submission.csv"
    CKPT_ROOT = DRIVE_ROOT / "Preprocessing_Output" / "checkpoints"  # atau lokasi lain hasil push checkpoint 02x
    ENSEMBLE_CONFIG_PATH = DRIVE_ROOT / "Preprocessing_Output" / "ensemble_config.json"
    CALIBRATION_PARAMS_PATH = DRIVE_ROOT / "Preprocessing_Output" / "calibration_params.json"
else:
    _raw_mount = resolve_kaggle_input_dir("bdc2026-raw")  # disediakan panitia; nama file sudah dinomori 1-1458
    TEST_ROOT = _raw_mount / "test"
    TEMPLATE_PATH = _raw_mount / "submission.csv"  # template resmi, urutan baris adalah hukum
    CKPT_ROOT = None  # tidak dipakai di Kaggle -- tiap checkpoint di-resolve per dataset slug-nya sendiri (lihat bawah)
    _nb03_mount = resolve_kaggle_input_dir("bdc2026-nb03-outputs")
    ENSEMBLE_CONFIG_PATH = _nb03_mount / "ensemble_config.json"
    CALIBRATION_PARAMS_PATH = _nb03_mount / "calibration_params.json"

TEST_IMAGE_EXT = "jpg"

# Gagal cepat & jelas kalau salah satu path input tidak ketemu, daripada FileNotFoundError samar
# di tengah loop inferensi. Penyebab paling umum di Kaggle: dataset gagal ter-attach ke kernel ini
# (kadang terjadi setelah error "ConcurrencyViolation / Failed to save draft" saat import notebook
# -- attachment dataset-nya tidak ikut ke-commit). Perbaikannya: cek panel "Input" di kanan, klik
# "Add Input" untuk attach ulang dataset yang hilang, lalu "Save Version" lagi.
for _check_path in [TEST_ROOT, TEMPLATE_PATH, ENSEMBLE_CONFIG_PATH, CALIBRATION_PARAMS_PATH]:
    if not _check_path.exists():
        raise FileNotFoundError(
            f"{_check_path} tidak ketemu (RUNTIME_ENV='{RUNTIME_ENV}'). Kalau di Kaggle: cek "
            f"panel 'Input', attach ulang dataset yang hilang lewat 'Add Input', lalu 'Save "
            f"Version' lagi. Kalau di Colab: pastikan Drive ter-mount dan NB03 sudah selesai jalan."
        )

OUTPUT_ROOT = Path("/kaggle/working") if RUNTIME_ENV == "kaggle" else DRIVE_ROOT / "Preprocessing_Output" / "submissions"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SUBMISSION_TAG = "Kepepet_Deadline"  # nama tim -- file akhir wajib bernama submission_NamaTim.csv (soal BDC bagian 4)

ARCH_REGISTRY = {
    "convnextv2_base": dict(
        timm_tag="convnextv2_base.fcmae_ft_in22k_in1k_384", drop_path_rate=0.3,
        ckpt_dataset_slug="bdc2026-ckpt-convnextv2",
    ),
    "swinv2_base_window12to24": dict(
        timm_tag="swinv2_base_window12to24_192to384.ms_in22k_ft_in1k", drop_path_rate=0.3,
        ckpt_dataset_slug="bdc2026-ckpt-swinv2",
    ),
    "maxvit_base_tf": dict(
        timm_tag="maxvit_base_tf_384.in21k_ft_in1k", drop_path_rate=0.3,
        ckpt_dataset_slug="bdc2026-ckpt-maxvit",
    ),
    "tf_efficientnetv2_m": dict(
        timm_tag="tf_efficientnetv2_m.in21k_ft_in1k", drop_path_rate=0.2,
        ckpt_dataset_slug="bdc2026-ckpt-effnetv2",
    ),
}


## Dataset test (PRD §8 — identitas numerik kanonis, path dibangun ulang dari integer + konstanta saja)

`image_id` adalah nama file numerik 1-1458 dari panitia langsung — skema penamaan-ulang kanonis
§6.2 sudah cocok dengan ini, jadi tidak perlu penamaan ulang di sini.


In [ ]:
class TestDataset(Dataset):
    def __init__(self, image_ids: np.ndarray, transform):
        self.image_ids = image_ids.astype(np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = int(self.image_ids[idx])
        path = f"{TEST_ROOT}/{image_id}.{TEST_IMAGE_EXT}"  # hanya integer + konstanta bersama (PRD 8)
        with Image.open(path) as im:
            im = im.convert("RGB")
            tensor = self.transform(im)
        return tensor, image_id


def build_eval_transform(img_size, mean, std, scale_ratio=1.0):
    # scale_ratio menggeser seberapa jauh "zoom" crop akhir terhadap gambar asli, TANPA mengubah
    # ukuran tensor akhir yang masuk model (tetap img_size x img_size di semua skala) -- aman
    # dipakai lintas arsitektur (CNN maupun ViT/Swin/MaxViT berbasis grid tetap), beda dengan
    # feed langsung resolusi berbeda yang berisiko error di model dengan window/patch grid tetap.
    resize_short_side = round(img_size / (0.95 * scale_ratio))
    return T.Compose([
        T.Resize(resize_short_side, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(img_size),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])


# v2: dari 2 view (identity+hflip, 1 skala) jadi 6 view (identity+hflip x 3 skala). Naik dari v1
# -- TIDAK byte-identical lagi dengan versi lama, wajib disertai regenerasi OOF ulang (skeleton 02x)
# sebelum dipakai bareng NB03, dan wajib byte-identical dengan salinannya di skeleton 02x (PRD 7.3, 7.8, 11).
TTA_SPEC_VERSION = "identity+hflip+3scale-softmax-mean-v2"
TTA_SCALE_RATIOS = [0.90, 1.0, 1.10]


@torch.no_grad()
def tta_predict_probs_multiscale(model, images_per_scale) -> torch.Tensor:
    """TTA multi-skala: len(TTA_SCALE_RATIOS) skala x {identity, horizontal flip}, rata-rata post-softmax."""
    probs_sum = None
    for images in images_per_scale:
        images = images.to(DEVICE, non_blocking=True)
        probs_identity = F.softmax(model(images), dim=1)
        probs_flip = F.softmax(model(torch.flip(images, dims=[3])), dim=1)
        view_sum = probs_identity + probs_flip
        probs_sum = view_sum if probs_sum is None else probs_sum + view_sum
    return probs_sum / (2 * len(images_per_scale))


def create_inference_model(timm_tag: str, drop_path_rate: float, img_size: int):
    # pretrained=False: bobotnya langsung ditimpa oleh best.ckpt hasil fine-tune di bawah;
    # pretrained_cfg (sehingga mean/std dari resolve_model_data_config) tidak terpengaruh flag ini.
    kwargs = dict(pretrained=False, num_classes=NUM_CLASSES, drop_path_rate=drop_path_rate, img_size=img_size)
    try:
        model = timm.create_model(timm_tag, **kwargs)
    except TypeError:
        kwargs.pop("img_size")
        model = timm.create_model(timm_tag, **kwargs)
    return model


def resolve_norm_stats(model):
    data_cfg = timm.data.resolve_model_data_config(model)
    return list(data_cfg["mean"]), list(data_cfg["std"])


## Enumerasi gambar test dari template resmi (bukan dari listing folder)

Kolom `id` dari template dipakai langsung dan satu-satunya sumber untuk set id + urutan baris;
baik loader maupun submission akhir mengacu padanya, sehingga tidak ada risiko ke-sort ulang
secara diam-diam.


In [ ]:
template_df = pd.read_csv(TEMPLATE_PATH)
assert list(template_df.columns[:2]) == ["id", "predicted"] or "id" in template_df.columns
template_ids = template_df["id"].values
assert len(template_ids) == 1458, f"seharusnya 1458 baris template, dapat {len(template_ids)}"
print(f"template: {len(template_ids)} baris, rentang id [{template_ids.min()}, {template_ids.max()}]")


## Inferensi TTA per-arsitektur, per-fold -> rata-rata 5 fold -> probabilitas arsitektur (PRD §11)


In [ ]:
arch_probs = {}  # arch -> array probabilitas (n_test, 3), sudah dirata-rata 5 fold

for arch, spec in ARCH_REGISTRY.items():
    fold_prob_sum = np.zeros((len(template_ids), NUM_CLASSES), dtype=np.float64)
    for k in range(N_FOLDS):
        if RUNTIME_ENV == "kaggle":
            ckpt_path = resolve_kaggle_input_dir(spec["ckpt_dataset_slug"]) / f"fold{k}" / "best.ckpt"
        else:
            ckpt_path = CKPT_ROOT / spec["ckpt_dataset_slug"] / f"fold{k}" / "best.ckpt"
        best = torch.load(ckpt_path, map_location="cpu")
        img_size = best["img_size"]  # resolusi final Stage-3 tiap arsitektur, dicatat saat training (PRD 7.7)

        model = create_inference_model(spec["timm_tag"], spec["drop_path_rate"], img_size).to(DEVICE)
        model.load_state_dict(best["weights"])
        model.eval()
        mean, std = resolve_norm_stats(model)

        # 1 DataLoader per skala TTA -- urutan id identik di semua loader (shuffle=False, dataset
        # sama), jadi aman di-zip lockstep per batch tanpa perlu re-match ulang lewat image_id.
        loaders = [
            DataLoader(TestDataset(template_ids, build_eval_transform(img_size, mean, std, ratio)),
                       batch_size=32, shuffle=False, drop_last=False, num_workers=2, pin_memory=True)
            for ratio in TTA_SCALE_RATIOS
        ]

        id_to_row = {int(v): i for i, v in enumerate(template_ids)}
        for batches in tqdm(zip(*loaders), desc=f"{arch} fold{k}", total=len(loaders[0]), leave=False):
            images_per_scale = [b[0] for b in batches]
            image_ids = batches[0][1]
            probs = tta_predict_probs_multiscale(model, images_per_scale).cpu().numpy()
            for image_id, p in zip(image_ids.numpy(), probs):
                fold_prob_sum[id_to_row[int(image_id)]] += p

        del model
        torch.cuda.empty_cache()

    arch_probs[arch] = fold_prob_sum / N_FOLDS  # rata-rata 5 fold -> probabilitas arsitektur
    print(f"{arch}: inferensi selesai untuk {len(template_ids)} gambar test")


## Weighted soft-vote (bobot §10.2) + kalibrasi (delta §10.3) -> argmax


In [ ]:
with open(ENSEMBLE_CONFIG_PATH) as f:
    ensemble_config = json.load(f)
with open(CALIBRATION_PARAMS_PATH) as f:
    calibration_params = json.load(f)

archs_ordered = ensemble_config["archs"]
final_weights = ensemble_config["final_weights"]
delta = np.array(calibration_params["delta"])

ens_probs = np.zeros((len(template_ids), NUM_CLASSES), dtype=np.float64)
for arch in archs_ordered:
    ens_probs += final_weights[arch] * arch_probs[arch]

log_probs = np.log(np.clip(ens_probs, 1e-12, 1.0))
final_preds = (log_probs + delta).argmax(axis=1)


## Validation gate (SEBELUM menulis file submission)

Tepat 1458 baris; set id dan urutan sama dengan template; `predicted` di {0,1,2}; tidak ada NaN.


In [ ]:
submission_df = pd.DataFrame({"id": template_ids, "predicted": final_preds})

assert len(submission_df) == 1458
assert np.array_equal(submission_df["id"].values, template_ids), "urutan baris menyimpang dari template — jangan pernah sort ulang"
assert submission_df["predicted"].isin([0, 1, 2]).all()
assert not submission_df.isna().any().any()
print("Validation gate LOLOS.")

output_path = OUTPUT_ROOT / f"submission_{SUBMISSION_TAG}.csv"
submission_df.to_csv(output_path, index=False)
print(f"Menulis {output_path}")


## Laporan sanity — distribusi kelas prediksi vs distribusi OOF terkalibrasi

Divergensi besar di sini artinya harus diinvestigasi SEBELUM submit (jatah submission cuma 3 kali,
skor tertinggi yang dihitung — PRD §1).


In [ ]:
test_dist = submission_df["predicted"].value_counts(normalize=True).sort_index().to_dict()
oof_dist = {int(k): v for k, v in calibration_params["predicted_class_distribution"].items()}

print("distribusi prediksi di test set:", {k: round(v, 4) for k, v in test_dist.items()})
print("distribusi OOF terkalibrasi:    ", {k: round(v, 4) for k, v in oof_dist.items()})
for c in range(NUM_CLASSES):
    diff = test_dist.get(c, 0.0) - oof_dist.get(c, 0.0)
    flag = "  <-- divergensi besar, investigasi" if abs(diff) > 0.05 else ""
    print(f"  kelas {c}: test={test_dist.get(c, 0.0):.4f} oof={oof_dist.get(c, 0.0):.4f} diff={diff:+.4f}{flag}")


## Disiplin submission (PRD §11)

Simpan satu submission "asuransi" lebih awal begitu ada model valid; sisakan <=2 submission untuk
ensemble terkalibrasi penuh di atas. Semua keputusan varian wajib mengacu bukti offline dari NB03
(`oof_report.md`) — tidak pernah leaderboard probing.


## §11.4 Laporan kesadaran-kebocoran post-hoc (opsional)

Hanya jalan SETELAH `submission_*.csv` di atas ditulis dan dibekukan — semua artefak upstream
(model, bobot, bobot ensemble, kalibrasi) sudah terkunci di titik ini, jadi perbandingan ini tidak
mungkin memengaruhi training, validasi, pemilihan model, atau tuning hyperparameter secara kausal.
Murni untuk pemahaman tim / laporan anomali ke panitia kalau memang diperlukan.


In [ ]:
RUN_POSTHOC_LEAKAGE_REPORT = False  # ubah ke True hanya setelah submission di atas final

if RUN_POSTHOC_LEAKAGE_REPORT:
    import imagehash

    assert output_path.exists(), "jalankan ini hanya setelah file submission ditulis dan dibekukan"

    train_processed_root = Path("/kaggle/input/bdc2026-master-data/processed")
    train_hashes = {
        p.stem: imagehash.phash(Image.open(p).convert("RGB"), hash_size=8)
        for p in sorted(train_processed_root.glob("*.jpg"))
    }
    test_hashes = {
        int(image_id): imagehash.phash(
            Image.open(TEST_ROOT / f"{image_id}.{TEST_IMAGE_EXT}").convert("RGB"), hash_size=8
        )
        for image_id in template_ids
    }

    near_dup_rows = []
    for test_id, test_hash in tqdm(test_hashes.items(), desc="posthoc phash"):
        for train_id, train_hash in train_hashes.items():
            if (test_hash - train_hash) <= 4:
                near_dup_rows.append({"test_image_id": test_id, "train_image_id": train_id,
                                       "hamming_distance": test_hash - train_hash})

    posthoc_df = pd.DataFrame(near_dup_rows)
    posthoc_df.to_csv(OUTPUT_ROOT / "posthoc_leakage_awareness_report.csv", index=False)
    print(f"laporan posthoc: {len(posthoc_df)} pasangan test/train near-duplicate ditemukan "
          f"(informational saja — submission sudah dibekukan sebelum sel ini jalan)")
